# Loan Collections Prioritiser: End-to-End Walkthrough
**Project Reference:** `SKY8-DS-FP-005` | **Roadmap:** `SKY8-DS-RM-005` (Core ML)
**Candidate:** Nahbila Faith
**Track:** Skye8 Company Ltd — Data Science and ML / AI Internship Track
**Context:** Microfinance Institutions & Credit Unions in Bamenda, Cameroon

---
## Executive Summary
Collections officers in microfinance typically sort delinquent borrowers by descending debt balance (*Biggest Debt First*).
This notebook walks through the machine learning pipeline that replaces this heuristic with an **Expected Recoverable Value** engine ($P(\text{cure}) \times \hat{r} \times \text{Debt}$), achieving **88.0% precision** (vs 7.0% incumbent) and **+2,164,156 FCFA (+65.6%)** in net cash recovery lift per 100 worked accounts.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Add project root to sys.path
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.data_loading import load_installments
from src.target import build_arrears_events, base_recovery_rate
from src.features import build_features, FEATURE_COLUMNS, FORBIDDEN_POST_CONTACT
from src.model import load_model, AdaptablePrioritiser, compute_xai_waterfall
from src.evaluate import evaluate_collections_policy
from src.contact import assign_bamenda_quarter, generate_bilingual_scripts, calculate_benskin_route_savings

print("✓ All Skye8 modules and dependencies successfully imported.")

## 1. Load Processed Production Call Book (400 Accounts)
Let us inspect the ranked production book generated for the 3 collections officers in Bamenda.

In [ ]:
data_file = project_root / "data" / "processed" / "weekly_400_accounts.csv"
df = pd.read_csv(data_file)
print(f"Loaded {len(df)} delinquent accounts across {len(df.columns)} multi-task features.")
cols_to_show = ["rank", "SK_ID_CURR", "assigned_officer", "bamenda_quarter", "overdue_amount", "recovery_probability", "partial_recovery_ratio", "recommended_channel", "expected_gross_recovery_fcfa"]
df[cols_to_show].head(10)

## 2. Quantitative Evaluation: Why Biggest Debt First Fails
Comparing the Skye8 Expected Value Prioritiser against the Incumbent heuristic on held-out test data.

In [ ]:
import json
metrics_file = project_root / "reports" / "metrics_summary.json"
with open(metrics_file) as f:
    metrics = json.load(f)

k100 = metrics["k_evaluations"]["100"]
print(f"ROC-AUC: {metrics['roc_auc']:.4f} | PR-AUC: {metrics['pr_auc']:.4f}")
print(f"Precision @ 100 Accounts: {k100['precision_ev']:.1%} vs {k100['precision_incumbent']:.1%} (Incumbent)")
print(f"Relative Lift in Cures: {k100['precision_gain_ratio']:.2f}x")
print(f"Net Cash Lift: +{k100['money_lift_fcfa']:,.0f} FCFA (+{k100['money_lift_pct']:.1f}%)")

## 3. Explainable AI (XAI) Waterfall for Single Borrower
Inspecting the local feature contributions driving Client #103380 score.

In [ ]:
sample = df.iloc[0]
xai = compute_xai_waterfall(sample)
print(f"Client #{sample['SK_ID_CURR']} - Score: {sample['recovery_probability']:.1%}")
print("\nTop Positive Factors:")
for p in xai["top_positive"]:
    print(f"  [+] {p['factor']} ({p['impact']}) -> {p['desc']}")
print("\nTop Negative Drag Factors:")
for n in xai["top_negative"]:
    print(f"  [-] {n['factor']} ({n['impact']}) -> {n['desc']}")

## 4. Culturally Authentic Bilingual Outreach Scripts
Inspecting the generated scripts in Formal English and Cameroon Pidgin English.

In [ ]:
eng_script, pid_script = generate_bilingual_scripts(
    account_id=int(sample["SK_ID_CURR"]),
    loan_id=int(sample["SK_ID_PREV"]),
    overdue_amount=float(sample["overdue_amount"]),
    assigned_officer=str(sample["assigned_officer"]),
    is_vulnerable=bool(sample["is_vulnerable"]),
    segment=str(sample["behavioral_segment"])
)
print("🇬🇧 FORMAL ENGLISH SCRIPT:")
print(eng_script)
print("\n🇨🇲 CAMEROON PIDGIN ENGLISH SCRIPT:")
print(pid_script)

## 5. Motorcycle Route Clustering & Benskin Fuel Economy
Evaluating the transit fuel savings for Officer 3 (Emmanuel K.) across Bamenda quarters.

In [ ]:
emmanuel_df = df[df["assigned_officer"].str.contains("Emmanuel", na=False)]
savings = calculate_benskin_route_savings(emmanuel_df)
print(f"Scheduled Field Visits: {savings['n_field_visits']}")
print(f"Unclustered Distance: {savings['unclustered_km']} km -> Clustered: {savings['clustered_km']} km")
print(f"Distance Cut: {savings['km_saved']} km (-{savings['pct_distance_saved']}%) ")
print(f"Fuel Saved: {savings['fuel_saved_liters']} Litres (@ 850 FCFA/L = {savings['fuel_cash_saved_fcfa']:,.0f} FCFA saved)")

## 6. Adaptable Prioritiser Demonstration
Demonstrating online Bayesian adaptation as collections officers record outcomes.

In [ ]:
engine = AdaptablePrioritiser()
print(f"Baseline Prior Odds Offset: {engine.branch_offset}")
engine.adapt_to_branch("Urban Commercial (Mankon / Commercial Ave)")
print(f"Adapted Urban Commercial Offset: {engine.branch_offset:+.2f}")
delta = engine.incorporate_field_feedback(int(sample["SK_ID_CURR"]), "Cured & Settled in Full")
print(f"Incorporated field feedback for Client #{sample['SK_ID_CURR']}: delta = {delta:+.1%}")